# 02 · 模型训练（Colab）

训练 `crn-nano` / `crn-lite` / `crn-large` 三档模型。

**为什么是三档而不是一个**：指标表里最有说服力的不是"我的模型 SI-SDR 多少"，
而是**帕累托前沿** —— 参数量/RTF 与质量之间的权衡曲线。
只有一个点画不出曲线，也就回答不了"要不要为 0.3 dB 多花 3 倍算力"这类真实的工程问题。

**训练结果全部保存到 Drive**（`DRIVE_ROOT/checkpoints/<模型名>/`）：
- `last.pt` —— 每个 epoch 覆盖写，用于断点续训
- `best.pt` —— 验证 SI-SDR 最好的那一版，导出用它
- `history.json` —— 训练曲线

Colab 会话随时可能断（12 小时上限、闲置回收、GPU 配额）。
checkpoint 里存的不只是权重，还有**优化器动量、学习率调度、随机数状态** ——
断线后重跑训练 cell 会自动续训，不会出现 loss 反弹。

In [ ]:
# ── 挂载 Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  配置 —— 所有路径与选择都集中在这一个 cell，别处不要再写死路径
# ═══════════════════════════════════════════════════════════════════════

# Drive 上的项目根。**持久**，会话结束不丢。
# 改成你实际存放 rtse-colab.zip 的目录。路径里有空格也没问题。
DRIVE_ROOT = '/content/drive/MyDrive/Audio AI/RTSE'

# Colab 本地盘。**临时**，会话结束即消失，但读写比 Drive 快得多。
WORK_ROOT = '/content/rtse_work'

# ── 原始语料（THCHS-30 / MUSAN / RIRS，合计约 23 GB）怎么放？────────────
#
#   'hybrid' —— **推荐**。压缩包缓存在 Drive，每次会话解压到本地盘。
#               一次下载永久有效；训练读取是本地盘全速；
#               每个新会话只需几分钟解压。Drive 占用约 23 GB。
#
#   'local'  —— 全部放本地盘，压缩包用完即删。
#               不占 Drive；代价是**每次新会话都要重下**（15~40 分钟）。
#
#   'drive'  —— 全部放 Drive，直接解压到 Drive。
#               ⚠ 不推荐：THCHS-30 有一万多个小文件，在 Drive 的 FUSE 挂载上
#               逐个创建极慢（每个文件都是一次 API 往返），解压可能要几小时；
#               训练时的随机读取也慢 2~5 倍。只有在 Colab 本地盘不够用时才选它。
DATA_MODE = 'hybrid'

# ── 快速验证模式 ────────────────────────────────────────────────────────
# True  = 只下载 337 MB 的小语料（LibriSpeech dev-clean，英文），
#         约 15 分钟就能把「数据→训练→导出→回传」整条链路跑通一遍。
#         **只用来验证流程，不要用它的结果做最终指标**（英文语料算不了中文 CER）。
# False = 完整流程（THCHS-30 中文 + MUSAN + RIRS，约 23 GB）
QUICK_TEST = False

# ═══════════════════════════════════════════════════════════════════════

import os, sys, json, shutil, subprocess, time
from pathlib import Path
from shlex import quote as shq          # 路径里有空格时，所有 shell 命令都要靠它

assert DATA_MODE in ('hybrid', 'local', 'drive'), f'DATA_MODE 只能是 hybrid/local/drive'

DRIVE = DRIVE_ROOT
WORK = WORK_ROOT

# 压缩包放哪 / 解压到哪 —— 三种模式的唯一区别就在这两行
ARCHIVE_DIR = f'{WORK}/archives' if DATA_MODE == 'local' else f'{DRIVE}/archives'
DATA = f'{DRIVE}/rawdata' if DATA_MODE == 'drive' else f'{WORK}/data'
KEEP_ARCHIVE = DATA_MODE != 'local'    # local 模式解压后删包省空间，其余保留以便复用

# Drive 侧的产物目录（**这些永远在 Drive 上**，训练结果不能放临时盘）
CKPT_DIR = f'{DRIVE}/checkpoints'   # 训练断点，每个 epoch 保存
MODEL_DIR = f'{DRIVE}/models'       # 导出的 ONNX
TESTSET_DIR = f'{DRIVE}/testset'    # 固定测试集
LOG_DIR = f'{DRIVE}/logs'

assert os.path.isdir(DRIVE), (
    f'Drive 上找不到 {DRIVE}\n'
    '检查两件事：① Drive 已挂载成功；② DRIVE_ROOT 与你实际的目录一致'
    '（区分大小写，路径里的空格照写即可）。'
)
for d in [WORK, ARCHIVE_DIR, DATA, CKPT_DIR, MODEL_DIR, TESTSET_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print('目录布局')
print('─' * 74)
print(f'  代码包(需手动上传)  {DRIVE}/rtse-colab.zip')
print(f'  压缩包缓存          {ARCHIVE_DIR}')
print(f'  语料解压目标        {DATA}')
print(f'  数据清单            {DRIVE}/manifest.json')
print(f'  固定测试集          {TESTSET_DIR}')
print(f'  训练断点  ★         {CKPT_DIR}/<模型名>/{{last,best}}.pt')
print(f'  导出模型  ★         {MODEL_DIR}/<模型名>.onnx')
print('─' * 74)
print(f'  数据模式  {DATA_MODE}' + {
    'hybrid': '   压缩包存 Drive（一次下载永久有效），每会话解压到本地盘',
    'local':  '   全在临时盘，每个新会话都要重新下载',
    'drive':  '   全在 Drive（解压会很慢，一万多个小文件走 FUSE）',
}[DATA_MODE])
print(f'  语料      {"快速验证(小语料/英文)" if QUICK_TEST else "完整流程(中文 THCHS-30)"}')
print()

!df -h /content | tail -1
!df -h /content/drive 2>/dev/null | tail -1

In [ ]:
# ── 安装项目代码 ────────────────────────────────────────────────────────
# rtse-colab.zip 由本地 `uv run python scripts/pack_for_colab.py` 生成，
# 需要手动上传到 DRIVE_ROOT 指向的目录。
ZIP = f'{DRIVE}/rtse-colab.zip'
assert os.path.exists(ZIP), (
    f'找不到 {ZIP}\n'
    '请先在本地执行 `uv run python scripts/pack_for_colab.py`，'
    f'再把 dist/rtse-colab.zip 上传到 Drive 的 {DRIVE} 下。\n'
    f'该目录下现有：{sorted(os.listdir(DRIVE))[:12]}'
)

SRC = f'{WORK}/rtse-src'
shutil.rmtree(SRC, ignore_errors=True)
os.makedirs(SRC, exist_ok=True)
!unzip -q -o {shq(ZIP)} -d {shq(SRC)}

# 只装项目需要而 Colab 没有预装的几个包。
# 不用 `pip install -e .`：那会去解析 pyproject 里锁定的 torch CPU 索引，
# 把 Colab 自带的 GPU 版 torch 覆盖掉 —— 训练会瞬间慢几十倍。
!pip install -q soxr pystoi jiwer webrtcvad-wheels pesq onnx onnxruntime 2>&1 | tail -2

sys.path.insert(0, f'{SRC}/src')
import rtse
print('rtse', rtse.__version__, '| SR', rtse.SAMPLE_RATE, '| n_fft', rtse.N_FFT, '| hop', rtse.HOP_LENGTH)

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# ── 自检：Colab 侧与本地必须是同一条信号链路 ───────────────────────────
# 这一步不能跳。如果 Colab 上的 STFT 与本地哪怕差一点，
# 训练出来的模型拿回本地就会掉点，而且极难定位（两边单独看都"没问题"）。
import numpy as np, torch
from rtse.audio.stft import stft, istft, check_cola, magnitude_db
from rtse.data.dataset import stft_torch, istft_torch

x = np.random.default_rng(0).standard_normal(16000)
print('COLA 偏差          :', f'{check_cola():.2e}')
print('numpy 完美重构      :', f'{np.max(np.abs(istft(stft(x), length=x.size) - x)):.2e}')

xt = torch.from_numpy(x).float().unsqueeze(0)
ref, got = stft(x), stft_torch(xt)
assert ref.shape[0] == got.shape[2], f'帧数不一致 {ref.shape[0]} vs {got.shape[2]}'
gc = got[0,0].numpy() + 1j*got[0,1].numpy()
print('torch/numpy STFT   :', f'{np.max(np.abs(gc - ref)) / np.max(np.abs(ref)):.2e} (相对)')
print('torch 往返重构      :', f'{(istft_torch(stft_torch(xt), length=16000) - xt).abs().max().item():.2e}')

t = np.arange(16000)/16000
db = magnitude_db(stft(np.sin(2*np.pi*1000*t))).max()
print('dBFS 标定(满幅正弦) :', f'{db:.3f} dB  (应为 0.000)')
assert abs(db) < 0.05, 'dBFS 标定不对，检查代码包是否为最新'
print('\n✅ Colab 与本地是同一条链路。')

## 1. 构建数据集

In [ ]:
from torch.utils.data import DataLoader
from rtse.data.dataset import OnlineMixDataset, MixConfig

mf = Path(f'{DRIVE}/manifest.json')
assert mf.exists(), f'找不到 {mf}，先跑 01_data_prep.ipynb'
manifest = json.loads(mf.read_text(encoding='utf-8'))
print(f'清单来源: {manifest["data_dir"]}   quick_test={manifest["quick_test"]}')

# 语料解压在临时盘时，新会话里文件已经没了 —— 提前查出来，别等训练跑一半才报错
probe = manifest['speech']['train'][0]
assert os.path.exists(probe), (
    f'清单里的语料文件不存在:\n  {probe}\n\n'
    '这是新会话、而语料解压在临时盘（DATA_MODE = hybrid/local）时的正常现象。\n'
    '解决：回到 01_data_prep.ipynb 重跑「下载语料」那个 cell。\n'
    f'  · hybrid 模式：压缩包还在 Drive，只会解压，几分钟\n'
    f'  · local  模式：需要重新下载，15~40 分钟\n'
    f'当前 DATA_MODE = {DATA_MODE}'
)

# OnlineMixDataset 直接接受**文件列表**（也接受目录）。
# 必须用清单而不是让它扫目录 —— 清单是按说话人划分好的，
# 扫目录会把测试说话人混进训练集，指标全部虚高。
MIX = MixConfig(segment_seconds=4.0, snr_range=(-5.0, 20.0), reverb_prob=0.5)

train_ds = OnlineMixDataset(manifest['speech']['train'], manifest['noise_train'],
                            manifest['rir_train'], cfg=MIX, length=20000, seed=0)
val_ds = OnlineMixDataset(manifest['speech']['val'], manifest['noise_test'],
                          manifest['rir_test'], cfg=MIX, length=800, seed=999)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2,
                      pin_memory=True, drop_last=True, persistent_workers=True)
val_dl = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# 抽一个 batch 看看，提前发现路径错、全静音之类的问题
from rtse.metrics.intrusive import si_sdr
nb_, cl_ = next(iter(train_dl))
print('batch', tuple(nb_.shape), '| 输入 SI-SDR 抽样:',
      [round(si_sdr(cl_[i].numpy(), nb_[i].numpy()), 1) for i in range(4)], 'dB')

## 2. 训练

**别照搬预估时间，看第一个 epoch 的实测值。** 训练器会把每个 epoch 的
`epoch_seconds` 记进 `history.json`，第一个 epoch 跑完就能算出总时长，
据此决定 `EPOCHS` 定多少、要分几次会话跑完。

关于 Colab 免费版（T4 + 2 vCPU）：

- **GPU 选 T4 就对了**，免费层没有更好的选项（A100 / L4 是 Pro 专属）。
  别选 TPU —— 本模型是 GRU + 因果卷积的流式结构，TPU 没有收益还要折腾 XLA。
  更别选 CPU，会慢几十倍。
- **数据加载不是瓶颈**（已实测：`reverb_prob=0.5` 下单样本 5.1 ms，
  2 worker 约 393 样本/秒，20000 样本的 epoch 只需不到 1 分钟）。
  所以 `num_workers=2` 够用，调大反而会在 2 vCPU 上互相抢占。
- 免费版会因**闲置**被回收（约 90 分钟无交互），也有动态用量上限。
  checkpoint 每个 epoch 都写 Drive，断了重跑本 cell 即可续训。

第一次建议把 `EPOCHS` 改成 5 跑一轮，确认 loss 在降、时间可接受，再改回 60。

In [ ]:
from rtse.models import build_model
from rtse.train import Trainer, TrainConfig

MODELS = ['crn-nano', 'crn-lite']      # 想跑大模型就加上 'crn-large'
EPOCHS = 60

for name in MODELS:
    out_dir = f'{CKPT_DIR}/{name}'     # ← 训练结果落在 Drive 上，会话断了也在
    os.makedirs(out_dir, exist_ok=True)
    cfg = TrainConfig(model=name, epochs=EPOCHS, batch_size=16, lr=3e-4,
                      out_dir=out_dir, num_workers=2, log_every=100)
    model = build_model(name)
    print(f'\n{"="*70}\n{name}   参数量 {model.count_params():,}   → {out_dir}\n{"="*70}')

    tr = Trainer(model, train_dl, val_dl, cfg)
    last = Path(out_dir) / 'last.pt'
    if last.exists():
        tr.load(last)                  # 断点续训（含优化器/调度/随机数状态）
    if tr.epoch >= EPOCHS:
        print(f'{name} 已完成（epoch {tr.epoch}），跳过'); continue
    tr.fit()

print('
训练产物（在 Drive 上，会话断了也在）：')
!ls -lh {shq(CKPT_DIR)}/*/

## 3. 训练曲线

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in MODELS:
    h = Path(f'{CKPT_DIR}/{name}/history.json')
    if not h.exists(): continue
    hist = json.loads(h.read_text())
    ep = [r['epoch'] for r in hist]
    axes[0].plot(ep, [r.get('loss') for r in hist], label=name)
    axes[1].plot(ep, [r.get('si_sdr') for r in hist], label=f'{name} train')
    if 'val_si_sdr' in hist[0]:
        axes[1].plot(ep, [r.get('val_si_sdr') for r in hist], '--', label=f'{name} val')
    axes[2].plot(ep, [r.get('spec') for r in hist], label=name)

for ax, t in zip(axes, ['总损失', 'SI-SDR (dB)', '压缩谱损失']):
    ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# 怎么读：
#   训练 SI-SDR 一路涨但验证走平 → 过拟合，加数据或加正则
#   两条都走平且数值低         → 欠拟合，或学习率有问题